In [ ]:
import numpy as np
import numpy.typing as npt
import adi
from scipy import signal
import threading

from config import config
from circular_buffer import WindowedCircularBuffer
from recording_buffer import RecordingBuffer
from save_queue import SaveQueue

In [ ]:
def producer(sdr: adi.Pluto, buffer: WindowedCircularBuffer) -> None:
    try:
        while buffer.is_active():
            # Get samples and write them to the buffer
            buffer.push_samples(sdr.rx())

    except Exception as e:
        print("\nProducer exited with the following error:\n", e)
        buffer.close()

In [ ]:
def writer(queue: SaveQueue) -> None:
    try:
        while True:
            # Save queued samples
            queue.dequeue_and_save()
    
    except Exception as e:
        raise Exception("\nWriter exited with the following error:\n", e)

In [ ]:
def detect_preamble(mag: npt.NDArray, preamble: npt.NDArray, std_thresh: float):
    # Correlate
    corr = np.correlate(mag, preamble, mode="valid")

    # Determine threshold
    thresh = np.mean(corr) + std_thresh * np.std(corr)

    # Find and return peaks
    peaks, _ = signal.find_peaks(corr, height=thresh, distance=len(preamble))
    verified_peaks = []
    for peak in peaks:
        preamble_data = mag[peak:peak + len(preamble)].reshape(-1, config.fs_mult).mean(axis=1)
        preamble_bits = preamble_data > (max(preamble_data) + min(preamble_data))/2

        # Ensure the preamble bits match the expected preamble
        if np.array_equal(config.preamble_mask, preamble_bits):
            verified_peaks.append(peak)

    return verified_peaks

In [ ]:
# Extend ADS-B preamble to match sampling rate
preamble_mask_long = np.repeat(config.preamble_mask, config.fs_mult)
preamble = np.zeros(len(preamble_mask_long), dtype=np.float64)
preamble[preamble_mask_long] = 2
preamble[~preamble_mask_long] = -1

# Initialise the SDR
sdr = adi.Pluto("ip:192.168.3.1")
sdr.gain_control_mode_chan0 = "slow_attack"
sdr.rx_lo = int(config.f)
sdr.sample_rate = int(config.fs)
sdr.rx_rf_bandwidth = int(config.bandwidth)
sdr.rx_buffer_size = config.READ_BLOCK_SIZE

# Create buffers
buffer = WindowedCircularBuffer(config.BUFFER_SIZE, config.WINDOW_SIZE)
signal_buffer = RecordingBuffer()
save_queue = SaveQueue()
remaining_samples = 0
recording = False

# Create writer thread
writer_thread = threading.Thread(
    target=writer,
    args=(save_queue)
)
writer_thread.start()

# Create producer thread
producer_thread = threading.Thread(
    target=producer,
    args=(sdr, buffer)
)
producer_thread.start()

# Main loop for streaming incoming data
try:
    while buffer.is_active():
        # Fetch window and calculate average power over the window
        window = buffer.peek_window()
        mag = np.abs(window)
        average_power = 10*np.log10(np.mean(mag**2))
        print(average_power)

        # If there is enough power where there could feasibly be a signal
        # or a signal is currently being recorded, continue processing
        if average_power > config.POWER_THRESH or remaining_samples > 0:
            # Conduct preamble correlation
            peaks = detect_preamble(mag, preamble, config.CORRELATION_STD_THRESH)

            # Start new recording
            if remaining_samples == 0 and len(peaks) > 0:
                # Record segement
                start = peaks[0]
                signal_buffer.append_window(window[start:])

                # Update remaining samples
                remaining_samples = config.sig_len - (config.WINDOW_SIZE - start)
                recording = True

            # Contiue prior recording
            elif remaining_samples > 0:
                # Record segment
                end = min(config.WINDOW_SIZE, config.OVERLAP + remaining_samples)
                signal_buffer.append_window(window[config.OVERLAP:end])

                # Update remaining samples
                remaining_samples -= end - config.OVERLAP


            ### Possibly extend recording if more peaks are detected ###


            # If finished recording, save current signal
            if remaining_samples <= 0 and recording:
                save_queue.enqueue(signal_buffer.get_buffer())
                signal_buffer.clear_buffer()
                recording = False
                
        buffer.pop_window(config.STEP)

finally:
    buffer.close()